**************** Decision Tree Implement from scratch ********************

In [45]:
import numpy as np


In [46]:
class Node():
    """
    A class representing a node in a decision tree.
    """

    def __init__(self, feature=None, threshold=None, left=None, right=None,
                 gain=None, value=None, entropy=None, class_counts=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.gain = gain
        self.value = value
        self.entropy = entropy        # entropy of this node's samples
        self.class_counts = class_counts  # {class: count} for leaf nodes

In [47]:
class DecisionTree():
    """
    A decision tree classifier for binary classification problems.
    """

    def __init__(self, min_samples=2, max_depth=2):
        self.min_samples = min_samples
        self.max_depth = max_depth
        
    def split_data(self, dataset, feature, threshold):
        left_dataset, right_dataset = [], []
        for row in dataset:
            if row[feature] <= threshold:
                left_dataset.append(row)
            else:
                right_dataset.append(row)
        return np.array(left_dataset), np.array(right_dataset)

    def entropy(self, y):
        entropy = 0
        for label in np.unique(y):
            p = len(y[y == label]) / len(y)
            entropy -= p * np.log2(p)
        return entropy

    def information_gain(self, parent, left, right):
        n_total = len(parent)
        weighted_child_entropy = (len(left) / n_total) * self.entropy(left) + \
                                 (len(right) / n_total) * self.entropy(right)
        return self.entropy(parent) - weighted_child_entropy

    def best_split(self, dataset, num_samples, num_features):
        best_split_gain = -1
        best_split = {}
        for feature_index in range(num_features):
            thresholds = np.unique(dataset[:, feature_index])
            for threshold in thresholds:
                left_dataset, right_dataset = self.split_data(dataset, feature_index, threshold)
                if len(left_dataset) and len(right_dataset):
                    y, left_y, right_y = dataset[:, -1], left_dataset[:, -1], right_dataset[:, -1]
                    information_gain = self.information_gain(y, left_y, right_y)
                    if information_gain > best_split_gain:
                        best_split_gain = information_gain
                        best_split["feature"] = feature_index
                        best_split["threshold"] = threshold
                        best_split["left_dataset"] = left_dataset
                        best_split["right_dataset"] = right_dataset
                        best_split["gain"] = information_gain
        return best_split

    def isBelowInfoGainThreshold(self, gain):
        return gain < 0.00001

    def build_tree(self, dataset, current_depth=0):
        num_samples, num_features = dataset.shape[0], dataset.shape[1] - 1
        yLabel = dataset[:, -1]
        node_entropy = self.entropy(yLabel)
        classes, counts = np.unique(yLabel, return_counts=True)
        class_counts = {int(c): int(n) for c, n in zip(classes, counts)}

        best = self.best_split(dataset, num_samples, num_features)

        if (num_samples < self.min_samples or
                current_depth >= self.max_depth or
                not best or
                self.isBelowInfoGainThreshold(best["gain"])):
            leaf_value = float(max(set(yLabel), key=list(yLabel).count))
            return Node(value=leaf_value, entropy=node_entropy, class_counts=class_counts)

        left  = self.build_tree(best["left_dataset"],  current_depth + 1)
        right = self.build_tree(best["right_dataset"], current_depth + 1)

        return Node(
            feature=best["feature"],
            threshold=best["threshold"],
            left=left,
            right=right,
            gain=best["gain"],
            entropy=node_entropy,
            class_counts=class_counts
        )

    def fit(self, X, y):
        dataset = np.concatenate((X, y), axis=1)
        self.root = self.build_tree(dataset)

    def predict(self, X):
        predictions = []
        for x in X:
            node = self.root
            while node.value is None:
                if x[node.feature] <= node.threshold:
                    node = node.left
                else:
                    node = node.right
            predictions.append(node.value)
        return predictions

In [ ]:
import numpy as np

# csv_file = '../data_part1/rtg_A.csv'
# csv_file = '../data_part1/rtg_B.csv'
csv_file = './data_part1/rtg_C.csv'

# Read header for feature names
with open(csv_file) as f:
    header = f.readline().strip().split(',')
feature_names = header[:-1]  # all columns except the last (class)

data = np.genfromtxt(csv_file, delimiter=',', skip_header=1)

X = data[:, :-1]
y = data[:, -1:]

# Train on ALL data — assignment focuses on tree structure, not test accuracy
tree_full = DecisionTree(min_samples=2, max_depth=10)
tree_full.fit(X, y)
predictions = tree_full.predict(X)

# Per-instance results
actuals = y.flatten()
accuracy = np.mean(np.array(predictions) == actuals)

print(f"Features: {feature_names}")
print(f"Overall Accuracy: {accuracy * 100:.1f}%  ({int(accuracy * len(actuals))}/{len(actuals)} correct)")

Features: ['att0', 'att1', 'att2', 'att3', 'att4', 'att5', 'att6']
Overall Accuracy: 100.0%  (50/50 correct)


In [49]:
np.random.seed(33)
indices = np.random.permutation(len(X))
split = int(0.7 * len(X))
X_train, X_test = X[indices[:split]], X[indices[split:]]
y_train, y_test = y[indices[:split]], y[indices[split:]]

# Train on Split data 
tree_split = DecisionTree(min_samples=2, max_depth=10)
tree_split.fit(X_train, y_train)
preds_split = tree_split.predict(X_test)
actuals_split = y_test.flatten()
accuracy_split = np.mean(np.array(preds_split) == actuals_split)

print(f"70/30 Test Accuracy: {accuracy_split * 100:.1f}%  ({int(accuracy_split * len(actuals_split))}/{len(actuals_split)})")

70/30 Test Accuracy: 60.0%  (9/15)


In [50]:
# ── Feature Importance Analysis ────────────────────────────────
def compute_all_feature_igs(dataset):
    """Compute initial IG for each feature at the root level."""
    num_features = dataset.shape[1] - 1
    y = dataset[:, -1]
    parent_entropy = tree_full.entropy(y)
    
    feature_igs = []
    for feature_idx in range(num_features):
        thresholds = np.unique(dataset[:, feature_idx])
        max_ig = 0
        best_threshold = None
        
        for threshold in thresholds:
            left_dataset, right_dataset = tree_full.split_data(dataset, feature_idx, threshold)
            if len(left_dataset) and len(right_dataset):
                left_y, right_y = left_dataset[:, -1], right_dataset[:, -1]
                ig = tree_full.information_gain(y, left_y, right_y)
                if ig > max_ig:
                    max_ig = ig
                    best_threshold = threshold
        
        feature_igs.append({
            'feature': feature_idx,
            'max_ig': max_ig,
            'best_threshold': best_threshold
        })
    
    return sorted(feature_igs, key=lambda x: x['max_ig'], reverse=True), parent_entropy

# Compute IGs
dataset = np.concatenate((X, y), axis=1)
feature_igs, root_entropy = compute_all_feature_igs(dataset)

# Track which features are actually used in the tree
def get_features_in_tree(node, used=set()):
    if node is None or node.value is not None:
        return used
    used.add(node.feature)
    get_features_in_tree(node.left, used)
    get_features_in_tree(node.right, used)
    return used

features_used = get_features_in_tree(tree_full.root)

for item in feature_igs:
    feat_idx = item['feature']
    feat_name = feature_names[feat_idx]
    max_ig = item['max_ig']
    best_thresh = item['best_threshold']
    used = "✓ YES" if feat_idx in features_used else "✗ NO"
    
    print(f"{feat_name:<12} {max_ig:<15.6f} {str(best_thresh):<15} {used:<15}")

print("=" * 70)
print(f"\nFeatures actually used in final tree: {[feature_names[i] for i in sorted(features_used)]}")
print(f"Features NOT used: {[feature_names[i] for i in sorted(set(range(X.shape[1])) - features_used)]}")

att6         0.085294        0.0             ✓ YES          
att4         0.058474        0.0             ✓ YES          
att3         0.044576        35.0            ✓ YES          
att1         0.038918        0.0             ✓ YES          
att5         0.038918        0.0             ✓ YES          
att0         0.028376        0.0             ✓ YES          
att2         0.002183        0.0             ✓ YES          

Features actually used in final tree: ['att0', 'att1', 'att2', 'att3', 'att4', 'att5', 'att6']
Features NOT used: []


In [51]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 1. Write tree to text file (depth-first traversal) ───────────────────────
def write_tree_txt(node, file, all_classes, feature_names, depth=0, edge_label=None):
    indent = "  " * depth
    if edge_label:
        file.write(f"{indent}{edge_label}\n")
        indent += "  "
    if node.value is not None:
        # Zero-fill missing classes
        counts = {int(c): node.class_counts.get(int(c), 0) for c in all_classes}
        counts_str = ", ".join(f"{c}: {counts[c]}" for c in sorted(counts))
        file.write(f"{indent}leaf {{{counts_str}}}\n")
    else:
        fname = feature_names[node.feature]
        file.write(f"{indent}{fname} (IG: {node.gain:.6f}, Entropy: {node.entropy:.6f})\n")
        write_tree_txt(node.left,  file, all_classes, feature_names, depth + 1, f"-- {fname} <= {node.threshold} --")
        write_tree_txt(node.right, file, all_classes, feature_names, depth + 1, f"-- {fname} > {node.threshold} --")

output_file = "output_tree_C.txt"
all_classes = sorted(np.unique(y.flatten()).astype(int))

# Track which features are actually used in the tree
def get_features_in_tree(node, used=set()):
    if node is None or node.value is not None:
        return used
    used.add(node.feature)
    get_features_in_tree(node.left, used)
    get_features_in_tree(node.right, used)
    return used

features_used = get_features_in_tree(tree_full.root)

with open(output_file, "w") as f:
    f.write(f"Decision Tree — trained on {len(actuals)} instances\n")
    f.write(f"Full Dataset Accuracy: {accuracy * 100:.1f}%\n")
    f.write(f"70/30 Split Test Accuracy: {accuracy_split * 100:.1f}%  ({int(accuracy_split * len(actuals_split))}/{len(actuals_split)})\n")
    f.write("=" * 60 + "\n\nTREE STRUCTURE (depth-first)\n" + "-" * 60 + "\n")
    write_tree_txt(tree_full.root, f, all_classes, feature_names)
    
    # Add feature importance analysis
    f.write("\n" + "=" * 60 + "\nFEATURE IMPORTANCE ANALYSIS\n" + "-" * 60 + "\n")
    f.write(f"{'Feature':<10} {'Max IG':<15} {'Used in Tree':<15}\n")
    f.write("-" * 40 + "\n")
    for item in feature_igs:
        feat_idx = item['feature']
        feat_name = feature_names[feat_idx]
        max_ig = item['max_ig']
        used = "✓ YES" if feat_idx in features_used else "✗ NO"
        f.write(f"{feat_name:<10} {max_ig:<15.6f} {used:<15}\n")
    f.write("-" * 40 + "\n")
    f.write(f"Features used: {[feature_names[i] for i in sorted(features_used)]}\n")
    f.write(f"Features NOT used: {[feature_names[i] for i in sorted(set(range(X.shape[1])) - features_used)]}\n")
    
    # Add per-instance results
    f.write("\n" + "=" * 60 + "\nPER-INSTANCE RESULTS\n" + "-" * 60 + "\n")
    f.write(f"{'Instance':>10} | {'Predicted':>10} | {'Actual':>10} | Result\n" + "-" * 52 + "\n")
    for i, (pred, actual) in enumerate(zip(predictions, actuals)):
        result = "CORRECT" if pred == actual else "WRONG"
        f.write(f"{i+1:>10} | {int(pred):>10} | {int(actual):>10} | {result}\n")
print(f"Tree written to: {output_file}")

Tree written to: output_tree_C.txt
